# 05 — Portfolio Systems

Model-based RL, PPO policy gradient, and Agentic RAG allocators.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.utils.seeds import set_all_seeds
set_all_seeds()
%matplotlib inline

## 1. Load Panel and Predictions

In [ ]:
from src.utils.io import load_parquet
from pathlib import Path
import yaml

with open('../config/assets.yaml') as f:
    assets = yaml.safe_load(f)

asset_universe = assets['sector_etfs'] + assets['indices'] + assets['commodities'] + assets['bond_etfs']
panel = load_parquet('../data/processed/panel.parquet')
hmm_probs = load_parquet('../data/regimes/hmm_probs.parquet')
print(f'Panel: {panel.shape if panel is not None else "not found"}')
print(f'HMM probs: {hmm_probs.shape if hmm_probs is not None else "not found"}')

## 2. Model-Based RL Allocator

In [ ]:
if panel is not None:
    from src.portfolio.model_based_rl import ModelBasedRLAllocator
    from src.evaluation.walk_forward import generate_folds, split_fold

    feature_cols = [c for c in panel.columns if c != 'target']
    dates = panel.index.get_level_values('date').unique()
    folds = generate_folds(dates, initial_train_years=5, test_months=1)

    n_assets = len(asset_universe)
    state_dim = len(feature_cols)

    allocator = ModelBasedRLAllocator(
        n_assets=n_assets,
        state_dim=state_dim,
        horizon=6,
        n_rollouts=50,
        epochs=20,
    )

    # Fit on first fold training data
    fold = folds[0]
    train_df, _ = split_fold(panel, fold)
    X_train = train_df[feature_cols].values
    # Build a returns matrix -- one row per date, one column per asset
    # (simplified: use target column as proxy for all assets here)
    y_train = np.column_stack([
        train_df['target'].values for _ in range(n_assets)
    ]) + np.random.default_rng(42).normal(0, 0.01, (len(train_df), n_assets))

    allocator.fit(X_train, y_train)
    example_state = X_train[-1]
    weights = allocator.allocate(example_state)
    print('Model-Based RL weights (example):', dict(zip(asset_universe[:len(weights)], weights.round(4))))
else:
    print('Panel not built -- run run_all.py first.')

## 3. PPO Policy Gradient Allocator

In [ ]:
if panel is not None:
    from src.portfolio.policy_gradient import PPOAllocator

    ppo = PPOAllocator(
        n_assets=n_assets,
        state_dim=state_dim,
        total_timesteps=10_000,  # reduced for demo; use 100_000 for full run
    )
    ppo.fit(X_train, y_train)
    weights_ppo = ppo.allocate(example_state)
    print('PPO weights (example):', dict(zip(asset_universe[:len(weights_ppo)], weights_ppo.round(4))))

## 4. Agentic RAG Allocator

> Requires ANTHROPIC_API_KEY to be set.

In [ ]:
import os
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY not set -- skipping agentic allocator.')
elif panel is None or hmm_probs is None:
    print('Panel or regime probs not found.')
else:
    from src.portfolio.agentic_rag import AgenticRAGAllocator
    from pathlib import Path

    # Load a sample prediction for each model to populate the store
    predictions_store = {}
    for p in sorted(Path('../results/predictions').glob('*.parquet')):
        df = load_parquet(p)
        if df is not None:
            predictions_store[p.stem] = df

    # Macro context: use the panel features for SPY
    spy_panel = panel.xs('SPY', level='ticker') if 'SPY' in panel.index.get_level_values('ticker') else None
    macro_df = spy_panel.drop(columns=['target'], errors='ignore') if spy_panel is not None else pd.DataFrame()

    agent = AgenticRAGAllocator(
        asset_universe=asset_universe,
        predictions_store=predictions_store,
        regime_probs=hmm_probs,
        macro_df=macro_df,
    )

    decision_date = hmm_probs.index[-1]
    weights_agent = agent.allocate(decision_date)
    print('Agent weights:', dict(zip(asset_universe[:len(weights_agent)], weights_agent.round(4))))